# EM-11 model comparison, notebook 2 of 2: score and compare

Scores every analysis window with YAMNet (CPU, as on the server) and AST (T4 GPUs), times each
model, re-runs the original 280-clip evaluation as a sanity check, then writes the comparison:
`results/report.md` and `results/report.json`.

**Settings (right-hand panel)**

- Accelerator: **GPU T4 x2**.
- Internet: **On** (code from GitHub, packages from PyPI, YAMNet from TF Hub, AST from Hugging Face).
- Input: **Add Input → Your Work → notebook 1's output** (the version you want). Do not attach the
  FSD50K dataset here.
- Environment: *Pin to original environment*.

Run it with **Save Version → Save & Run All (Commit)**. With notebook 1's smoke output it takes
about 20-30 minutes; with the full output, a few hours (the smoke run prints a projection).
Send back `results/report.md`, `results/report.json` and `results/verify-report.md`.

In [ ]:
REF = None      # None: the commit notebook 1 ran (its code.json); or a branch, tag or commit
SAMPLE = 2000   # windows AST also scores in half precision, to check that half precision changes nothing

In [ ]:
import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from IPython.display import Markdown, display


def sh(*cmd, check=True, cwd=None):
    """Run one command, its output shown here (a child's own output would not reach the notebook)."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() and check:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc.returncode


def out(*cmd, cwd=None):
    return subprocess.check_output([str(c) for c in cmd], text=True, cwd=cwd).strip()


def find(name, root=Path("/kaggle/input"), depth=4, dirs=False):
    """Paths under /kaggle/input whose name contains ``name``, nearest first."""
    hits = [p for d in range(1, depth + 1) for p in root.glob("/".join("*" * d))
            if name in p.name.lower() and (p.is_dir() or not dirs)]
    return sorted(hits, key=lambda p: len(p.parts))


UV = [sys.executable, "-m", "uv"]


def venv(name, requirements):
    """A clean Python 3.12 environment with pinned packages; what it holds is saved next to the results."""
    path = Path(f"/tmp/venv-{name}")
    shutil.rmtree(path, ignore_errors=True)
    sh(*UV, "venv", "--quiet", "--python", "3.12", path)
    sh(*UV, "pip", "install", "--quiet", "--python", path / "bin/python", "-r", CODE / requirements)
    freeze = out(*UV, "pip", "freeze", "--python", path / "bin/python")
    (WORK / f"env-{name}.txt").write_text(freeze + "\n")
    return path / "bin/python"

WORK = Path("/kaggle/working")
CODE, DATA, WEIGHTS = Path("/tmp/encoremoment"), Path("/tmp/em11-data"), Path("/tmp/ast")
BANKS, TIMING, RESULTS, LOGS = (WORK / d for d in ("banks", "timing", "results", "logs"))
for d in (BANKS, TIMING, RESULTS, LOGS):
    d.mkdir(parents=True, exist_ok=True)
os.environ["TFHUB_CACHE_DIR"] = "/tmp/tfhub"
PHASES = {}


def parallel(phase, jobs, every=600):
    """Run (name, command) jobs at once, each logging to logs/NAME.log; if one fails, stop all."""
    started, running = time.time(), {}
    for name, cmd in jobs:
        log = open(LOGS / f"{name}.log", "w")
        running[name] = (subprocess.Popen([str(c) for c in cmd], stdout=log, stderr=subprocess.STDOUT, cwd=CODE), log)
        print("started", name, flush=True)
    last = time.time()
    while running:
        time.sleep(10)
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            tail = (LOGS / f"{name}.log").read_text().strip().splitlines()[-1:]
            print(f"{name}: exit {proc.returncode} {tail}", flush=True)
            if proc.returncode:
                for p, _ in running.values():
                    p.terminate()
                raise RuntimeError(f"{name} failed: see logs/{name}.log")
        if running and time.time() - last > every:
            last = time.time()
            for name in running:
                lines = (LOGS / f"{name}.log").read_text().strip().splitlines()
                print(f"  {name}: {lines[-1] if lines else '...'}", flush=True)
    PHASES[phase] = round((time.time() - started) / 60, 1)
    print(f"{phase}: {PHASES[phase]} minutes", flush=True)

## Data from notebook 1, code and environment

In [ ]:
tars = find("em11-data.tar")
if len(tars) != 1:
    raise SystemExit(f"Add Input: notebook 1's output (found {tars})")
shutil.rmtree(DATA, ignore_errors=True)
sh("tar", "-xf", tars[0], "-C", DATA.parent)
manifest = json.loads((DATA / "manifest.json").read_text())
code = json.loads((DATA / "code.json").read_text())
SMOKE = manifest["limit"] is not None
print(f"data from {tars[0]}: {len(manifest['files'])} clips, created {manifest['created']}, "
      f"copy used: {manifest['checks']['copy_used']}, smoke: {SMOKE}")

shutil.rmtree(CODE, ignore_errors=True)
sh("git", "clone", "--quiet", code["repo"], CODE)
sh("git", "-C", CODE, "checkout", "--quiet", REF or code["commit"])
COMMIT = out("git", "-C", CODE, "rev-parse", "HEAD")
print("code at", COMMIT, "(notebook 1 ran", code["commit"] + ")")

sh(sys.executable, "-m", "pip", "install", "--quiet", "uv==0.9.5")
TF = venv("tf", "server/requirements-yamnet.txt")
AST = venv("ast", "server/tools/kaggle/requirements-ast.txt")

## GPUs and model weights (pinned, checked)

In [ ]:
sh("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")
GPUS = int(out(AST, "-c", "import torch; print(torch.cuda.device_count())"))
if GPUS < 1:
    raise SystemExit("Accelerator: GPU T4 x2")
print(GPUS, "GPUs,", os.cpu_count(), "CPUs")

PINS = json.loads(out(AST, "-c", "import json, sys; sys.path.insert(0, 'server/tools'); import em11_compare; "
                      "print(json.dumps(em11_compare.PINS))", cwd=CODE))
pin = PINS["ast"]
sh(AST, "-c", "from huggingface_hub import snapshot_download; "
   f"snapshot_download({pin['repo']!r}, revision={pin['revision']!r}, local_dir={str(WEIGHTS)!r}, "
   f"allow_patterns=['*.json', {pin['file']!r}])")
digest = out("sha256sum", WEIGHTS / pin["file"]).split()[0]
assert digest == pin["sha256"], f"AST weights {digest}, expected {pin['sha256']}"
print("AST", pin["repo"], pin["revision"][:12], "sha256 ok")
# YAMNet from TF Hub; the score stage checks its digest against the pin.
sh(TF, "-c", "import sys; sys.path.insert(0, 'server'); from encore_server.models import YamnetModel; "
   "print(YamnetModel().name)", cwd=CODE)

## Score

Each model's scores for every distinct window go into `banks/` (kept with this version, so the
analysis can be re-run without a GPU). AST jobs split the windows between the GPUs; YAMNet runs on
one CPU thread beside them. Half precision is used only for AST on the everyday clips, the largest
and cheapest-to-get-wrong set; a sample is scored both ways and compared.

In [ ]:
TOOL = CODE / "server/tools/em11_compare.py"
ALL = "curated,mixed,dev,everyday"


def score(py, variant, name, sets, *more):
    return [py, TOOL, "score", "--data", DATA, "--model", variant, "--sets", sets, "--out", BANKS / f"{name}.npz", *more]


def ast(variant, name, sets, gpu, shards=None, precision="fp32", *more):
    return score(AST, variant, name, sets, "--weights", WEIGHTS, "--device", f"cuda:{gpu}", "--precision", precision,
                 "--shard", f"{gpu}/{shards or GPUS}", "--threads", 1, *more)


def banks(*patterns):
    return ",".join(str(BANKS / p.format(g)) for p in patterns for g in range(GPUS))


parallel("yamnet + ast-pad10s fp32",
         [("yamnet", score(TF, "yamnet", "yamnet", ALL, "--threads", 1))]
         + [(f"pad10s-fp32-{g}", ast("ast-pad10s", f"pad10s-fp32-{g}", "curated,mixed,dev", g)) for g in range(GPUS)])
skip = [a for f in banks("pad10s-fp32-{}.npz").split(",") for a in ("--skip", f)]
parallel("ast-pad10s fp16 everyday",
         [(f"pad10s-fp16-{g}", ast("ast-pad10s", f"pad10s-fp16-{g}", "everyday", g, None, "fp16", *skip)) for g in range(GPUS)])
parallel("ast-cut1s", [(f"cut1s-{g}", ast("ast-cut1s", f"cut1s-{g}", ALL, g)) for g in range(GPUS)])
parallel("ast-clip", [(f"clip-{g}", ast("ast-clip", f"clip-{g}", "curated,dev,everyday", g)) for g in range(GPUS)])
parallel("fp16 sample", [("pad10s-fp16-sample", ast("ast-pad10s", "pad10s-fp16-sample", "curated", 0, 1, "fp16", "--sample", SAMPLE))])

## Time per window

One window at a time, nothing else running: YAMNet on 2 CPU threads (the cloud server) and on all;
AST on a T4 and on 2 CPU threads.

In [ ]:
def timing(py, variant, name, device, *more):
    return [py, TOOL, "timing", "--data", DATA, "--model", variant, "--device", device, "--out", TIMING / f"{name}.json", *more]


started = time.time()
for cmd in (timing(TF, "yamnet", "yamnet-cpu2", "cpu", "--threads", 2),
            timing(TF, "yamnet", "yamnet-cpu", "cpu"),
            timing(AST, "ast-pad10s", "pad10s-t4", "cuda:0", "--weights", WEIGHTS),
            timing(AST, "ast-pad10s", "pad10s-t4-fp16", "cuda:0", "--weights", WEIGHTS, "--precision", "fp16"),
            timing(AST, "ast-cut1s", "cut1s-t4", "cuda:0", "--weights", WEIGHTS),
            timing(AST, "ast-cut1s", "cut1s-cpu2", "cpu", "--weights", WEIGHTS, "--threads", 2),
            timing(AST, "ast-pad10s", "pad10s-cpu2", "cpu", "--weights", WEIGHTS, "--threads", 2, "--windows", 20)):
    sh(*cmd, cwd=CODE)
PHASES["timing"] = round((time.time() - started) / 60, 1)

## Sanity check

`em11_eval.py` on the original 280 clips (detector-v3), from this run's 16 kHz files, against the
rows the laptop produced (`docs/em-11-compare-sanity.json`). Skipped in a smoke run.

In [ ]:
if SMOKE:
    print("smoke run: no sanity check (the 280 clips are not prepared)")
else:
    started = time.time()
    rc = sh(TF, TOOL, "sanity", "--data", DATA, "--work", "/tmp/sanity", "--out", RESULTS / "sanity.json", check=False, cwd=CODE)
    print("every row matches" if rc == 0 else "rows differ: see results/sanity.json (the report shows them too)")
    PHASES["sanity"] = round((time.time() - started) / 60, 1)

## Compare

In [ ]:
started = time.time()
args = ["--bank", f"yamnet={BANKS / 'yamnet.npz'}",
        "--bank", "ast-pad10s=" + banks("pad10s-fp32-{}.npz", "pad10s-fp16-{}.npz"),
        "--bank", "ast-cut1s=" + banks("cut1s-{}.npz"),
        "--bank", "ast-clip=" + banks("clip-{}.npz"),
        "--check", f"ast-pad10s={BANKS / 'pad10s-fp16-sample.npz'}",
        *[a for p in sorted(TIMING.glob("*.json")) for a in ("--timing", p)]]
if (RESULTS / "sanity.json").exists():
    args += ["--sanity", RESULTS / "sanity.json"]
sh(AST, TOOL, "analyze", "--data", DATA, "--out", RESULTS, "--commit", COMMIT, "--workers", os.cpu_count(), *args, cwd=CODE)
PHASES["analyze"] = round((time.time() - started) / 60, 1)
shutil.copy(DATA / "verify-report.md", RESULTS / "verify-report.md")
for f in WORK.glob("env-*.txt"):
    shutil.move(f, RESULTS / f.name)
display(Markdown((RESULTS / "report.md").read_text()))

## Run summary (minutes per phase; windows per second per bank)

In [ ]:
print(json.dumps(PHASES, indent=1))
import numpy as np  # the notebook's own numpy, only to read bank metadata
for f in sorted(BANKS.glob("*.npz")):
    with np.load(f) as z:
        m = json.loads(str(z["meta"]))
    print(f"{f.name}: {m['rows']} rows, {m['per_second']} per second, {m['wall_seconds']} s")
if SMOKE:
    print("A full run scores about 400,000 distinct windows per window model; divide by the rates above "
          "(both GPUs together) for a rough time.")
(RESULTS / "run.json").write_text(json.dumps({"commit": COMMIT, "data": code, "phases": PHASES, "gpus": GPUS}, indent=1) + "\n")